In [ ]:
%load_ext autoreload
%autoreload 2
from noisepy.seis import correlate, stack_cross_correlations, __version__       # noisepy core functions
from noisepy.seis.io.asdfstore import ASDFCCStore, ASDFStackStore          # Object to store ASDF data within noisepy
from noisepy.seis.io.s3store import SCEDCS3DataStore # Object to query SCEDC data from on S3
from noisepy.seis.io.channel_filter_store import channel_filter
from noisepy.seis.io.datatypes import CCMethod, ConfigParameters, FreqNorm, RmResp, StackMethod, TimeNorm        # Main configuration object
from noisepy.seis.io.channelcatalog import XMLStationChannelCatalog        # Required stationXML handling object
import os
import shutil
from datetime import datetime, timezone
from datetimerange import DateTimeRange


from noisepy.seis.io.plotting_modules import plot_all_moveout

print(f"Using NoisePy version {__version__}")

path = "./scedc_data" 

os.makedirs(path, exist_ok=True)
cc_data_path = os.path.join(path, "CCF")
stack_data_path = os.path.join(path, "STACK")
S3_STORAGE_OPTIONS = {"s3": {"anon": True}}

In [ ]:
# SCEDC S3 bucket common URL characters for that day.
S3_DATA = "s3://scedc-pds/continuous_waveforms/"
# timeframe for analysis
start = datetime(2002, 1, 1, tzinfo=timezone.utc)
end = datetime(2002, 1, 2, tzinfo=timezone.utc)
timerange = DateTimeRange(start, end)
print(timerange)

S3_STATION_XML = "s3://scedc-pds/FDSNstationXML/CI/"            # S3 storage of stationXML

In [ ]:
# Initialize ambient noise workflow configuration
config = ConfigParameters() # default config parameters which can be customized

In [ ]:
config.start_date = start
config.end_date = end
config.acorr_only = False # only perform auto-correlation or not
config.xcorr_only = True # only perform cross-correlation or not

config.inc_hours = 24 # INC_HOURS is used in hours (integer) as the 
        #chunk of time that the paralelliztion will work.
        # data will be loaded in memory, so reduce memory with smaller 
        # inc_hours if there are over 400+ stations.
        # At regional scale for SCEDC, we can afford 20Hz data and inc_hour 
        # being a day of data.

 
# pre-processing parameters
config.samp_freq= 20  # (int) Sampling rate in Hz of desired processing (it can be different than the data sampling rate)
config.cc_len= 3600  # (float) basic unit of data length for fft (sec)
config.step= 1800.0  # (float) overlapping between each cc_len (sec)

config.ncomp = 3  # 1 or 3 component data (needed to decide whether do rotation)

config.stationxml= False  # station.XML file used to remove instrument response for SAC/miniseed data
      # If True, the stationXML file is assumed to be provided.
config.rm_resp= RmResp.INV  # select 'no' to not remove response and use 'inv' if you use the stationXML,'spectrum',


############## NOISE PRE-PROCESSING ##################
config.freqmin,config.freqmax = 0.08,10.0  # broad band filtering of the data before cross correlation
config.max_over_std  = 10  # threshold to remove window of bad signals: set it to 10*9 if prefer not to remove them

################### SPECTRAL NORMALIZATION ############
config.freq_norm= FreqNorm.RMA  # choose between "rma" for a soft whitening or "no" for no whitening. Pure whitening is not implemented correctly at this point.
config.smoothspect_N = 10  # moving window length to smooth spectrum amplitude (points)
    # here, choose smoothspect_N for the case of a strict whitening (e.g., phase_only)


#################### TEMPORAL NORMALIZATION ##########
config.time_norm = TimeNorm.ONE_BIT # 'no' for no normalization, or 'rma', 'one_bit' for normalization in time domain,
config.smooth_N= 10  # moving window length for time domain normalization if selected (points)


############ cross correlation ##############

config.cc_method= CCMethod.XCORR # 'xcorr' for pure cross correlation OR 'deconv' for deconvolution;
    # FOR "COHERENCY" PLEASE set freq_norm to "rma", time_norm to "no" and cc_method to "xcorr"

# OUTPUTS:
config.substack = True  # True = smaller stacks within the time chunk. False: it will stack over inc_hours
config.substack_len = config.cc_len  # how long to stack over (for monitoring purpose): need to be multiples of cc_len
    # if substack=True, substack_len=2*cc_len, then you pre-stack every 2 correlation windows.
    # for instance: substack=True, substack_len=cc_len means that you keep ALL of the correlations
    # if substack=False, the cross correlation will be stacked over the inc_hour window

### For monitoring applications ####
## we recommend substacking ever 2-4 cross correlations and storing the substacks
# e.g. 
# config.substack = True 
# config.substack_len = 4* config.cc_len

config.maxlag= 60  # lags of cross-correlation to save (sec)

In [ ]:
# For this tutorial make sure the previous run is empty
#os.system(f"rm -rf {cc_data_path}")
if os.path.exists(cc_data_path):
    shutil.rmtree(cc_data_path)

In [ ]:
#stations = "RPV,STS,LTP,LGB,WLT,CPP,PDU,CLT,SVD,BBR".split(",") # filter to these stations
stations = "RPV,SVD,BBR".split(",") # filter to these stations
# stations = "DGR,DEV,DLA,DNR,FMP,HLL,LGU,LLS,MLS,PDU,PDR,RIN,RIO,RVR,SMS,BBR,CHN,MWC,RIO,BBS,RPV,ADO,DEV".split(",") # filter to these stations

# There are 2 ways to load stations: You can either pass a list of stations or load the stations from a text file.
# TODO : will be removed with issue #270
config.load_stations(stations)

# For loading it from a text file, write the path of the file in stations_file field of config instance as below
# config.stations_file = os.path.join(os.path.dirname(__file__), "path/my_stations.txt")

# TODO : will be removed with issue #270
# config.load_stations()

catalog = XMLStationChannelCatalog(S3_STATION_XML, storage_options=S3_STORAGE_OPTIONS) # Station catalog
raw_store = SCEDCS3DataStore(S3_DATA, catalog, 
                             channel_filter(config.net_list, config.stations, ["BHE", "BHN", "BHZ"]), 
                             timerange, storage_options=S3_STORAGE_OPTIONS) # Store for reading raw data from S3 bucket
cc_store = ASDFCCStore(cc_data_path) # Store for writing CC data

In [ ]:
span = raw_store.get_timespans()
print(span)

In [ ]:
channel_list=raw_store.get_channels(span[0])
print(channel_list)

In [ ]:
d=raw_store.read_data(span[0],channel_list[0])
sampling_rate=d.sampling_rate

In [ ]:
d.stream.plot()

In [ ]:
import numpy as np
from obspy.signal.filter import bandpass

freqmin = config["freqmin"]
freqmax = config["freqmax"]
samp_freq = config["samp_freq"]
sps = sampling_rate

# parameters for butterworth filter
f1 = 0.9 * freqmin
f2 = freqmin
if 1.1 * freqmax > 0.45 * samp_freq:
    f3 = 0.4 * samp_freq
    f4 = 0.45 * samp_freq
else:
    f3 = freqmax
    f4 = 1.1 * freqmax
pre_filt = [f1, f2, f3, f4]

d.stream.taper(max_percentage=0.05, max_length=50)  # taper window
d.stream = np.float32(bandpass(d.stream , pre_filt[0], pre_filt[-1], df=sps, corners=4, zerophase=True))
print( pre_filt[0], pre_filt[-1])

In [ ]:
from noisepy.seis.noise_module import cut_trace_make_stat
trace_stdS, dataS_t, dataS = cut_trace_make_stat(config, d)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(dataS_t, dataS)
plt.show()

In [ ]:
import numpy as np
from noisepy.seis.noise_module import whiten, moving_ave
from noisepy.seis.io.datatypes import ConfigParameters, FreqNorm, TimeNorm
import scipy
from scipy.fftpack import next_fast_len

def noise_processing(fft_para: ConfigParameters, dataS):
    """
    this function performs time domain and frequency domain normalization if needed. in real case, we prefer use include
    the normalization in the cross-correaltion steps by selecting coherency or decon
    (Prieto et al, 2008, 2009; Denolle et al, 2013)
    PARMAETERS:
    ------------------------
    fft_para: ConfigParameters class containing all useful variables used for fft and cc
    dataS: 2D matrix of all segmented noise data
    # OUTPUT VARIABLES:
    source_white: 2D matrix of data spectra
    """
    # ------to normalize in time or not------
    if fft_para.time_norm != TimeNorm.NO:
        if fft_para.time_norm == TimeNorm.ONE_BIT:  # sign normalization
            white = np.sign(dataS)
        elif fft_para.time_norm == TimeNorm.RMA:  # running mean: normalization over smoothed absolute average
            white = np.zeros(shape=dataS.shape, dtype=dataS.dtype)
            for kkk in range(dataS.shape[0]):
                white[kkk, :] = dataS[kkk, :] / moving_ave(np.abs(dataS[kkk, :]), fft_para.smoothspect_N)

    else:  # don't normalize
        white = dataS

    # -----to whiten or not------
    if fft_para.freq_norm != FreqNorm.NO:
        source_white = whiten(white, fft_para)  # whiten and return FFT
    else:
        Nfft = int(next_fast_len(int(dataS.shape[1])))
        source_white = scipy.fftpack.fft(white, Nfft, axis=1)  # return FFT

    return source_white

In [ ]:
config.freq_norm= FreqNorm.NO
config.smoothspect_N = 10
config.time_norm = TimeNorm.ONE_BIT 
config.smooth_N= 10

# check 1 
trace_stdS, dataS_t, dataS = cut_trace_make_stat(config, d)
N = dataS.shape[0]
data_whiten=noise_processing(config, dataS)

Fs =  sampling_rate # Sampling frequency
T = 1 / Fs  # Sampling interval
nu0 = len(dataS[0])
freq = np.fft.fftfreq(nu0, d=T)

plt.xscale('log')
plt.yscale('log')
plt.xlim(0.01,12)
plt.plot(freq, np.abs(data_whiten.T),'-', linewidth=0.5)
plt.show()


In [ ]:
config.freq_norm= FreqNorm.RMA
config.smoothspect_N = 2
config.time_norm = TimeNorm.ONE_BIT 
config.smooth_N= 10

# check 1 
trace_stdS, dataS_t, dataS = cut_trace_make_stat(config, d)
N = dataS.shape[0]
data_whiten=noise_processing(config, dataS)

Fs =  sampling_rate # Sampling frequency
T = 1 / Fs  # Sampling interval
nu0 = len(dataS[0])
freq = np.fft.fftfreq(nu0, d=T)

plt.xscale('log')
plt.yscale('log')
plt.xlim(0.01,12); plt.ylim(0.01,10)
plt.plot(freq, np.abs(data_whiten.T),'-', linewidth=0.5)
plt.show()


In [ ]:
Nfft = data_whiten.shape[1]
Nfft2 = Nfft // 2

# load fft data in memory for cross-correlations
data = data_whiten[:, :Nfft2]
fft = data.reshape(data.size)
std = trace_stdS
fft_time = dataS_t

In [ ]:
from noisepy.seis.io.datatypes import NoiseFFT
from noisepy.seis.io.utils import get_results
from collections import OrderedDict
from typing import Dict


fft_refs = [NoiseFFT(fft, std, fft_time, N, Nfft)]
#fft_datas = get_results(fft_refs, "Compute ffts")
fft_datas = fft_refs

# Dictionary to store all the FFTs, keyed by channel index
ffts: Dict[int, NoiseFFT] = OrderedDict()
for ix_ch, fft_data in enumerate(fft_datas):
    if fft_data.fft.size > 0:
        ffts[ix_ch] = fft_data

Nfft = max(map(lambda d: d.length, fft_datas))

In [ ]:
print(ffts)

In [ ]:
print(channel_list)
src_chan=0
rec_chan=0
channels=channel_list


In [ ]:

print(src_chan, rec_chan, channels)
print(ffts)
print(Nfft)

In [ ]:
result = correlate.cross_correlation(config, src_chan, rec_chan, channels, ffts, Nfft)

In [ ]:
print(result[3])
print(result[3].shape)
print(result[2].get('dt'))
print(result[2].items())
deltat=result[2].get('dt')
maxlag=result[2].get('maxlag')

xtlag= np.arange(-maxlag, (maxlag)+deltat, deltat)
print(xtlag)

In [ ]:
plt.plot(xtlag, result[3].T)
plt.show()

In [ ]:
print(result[3].shape)
print(result[3].dtype)

In [ ]:
stackuu=np.asarray(result[3])
print(stackuu.shape)
dt=deltat

k=15
du0=stackuu[k]
du1=stackuu[k+1]
du2=stackuu[k+2]

# Generate or load your signal
# Example: Generate a sine wave signal
Fs =  1/dt # Sampling frequency
T = 1 / Fs  # Sampling interval
t = np.arange(0, 1, T)  # Time vector
f = 5  # Frequency of the signal

# Compute the FFT
fft_result = np.fft.fft(du0)
nu0 = len(du0)
freq = np.fft.fftfreq(nu0, d=T)
mag0 = np.abs(fft_result)

# Compute the FFT
fft_result = np.fft.fft(du1)
nu1 = len(du1)
freq1 = np.fft.fftfreq(nu1, d=T)
mag1 = np.abs(fft_result)

# Compute the FFT
fft_result = np.fft.fft(du2)
nu2 = len(du2)
freq2 = np.fft.fftfreq(nu2, d=T)
mag2 = np.abs(fft_result)

# Plot the magnitude spectrum in log-log scale
plt.figure(figsize=(10, 6))
plt.plot(freq[:nu0//2], mag0[:nu0//2]/nu0, label=f'signal seg {k}')
plt.plot(freq1[:nu1//2], mag1[:nu1//2]/nu1, label=f'signal seg {k+1}')
plt.plot(freq2[:nu2//2], mag2[:nu2//2]/nu2, label=f'signal seg {k+2}')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Test Signal Spectrum')
plt.legend()
plt.grid(True)
plt.show()